
##  Анализ датасета MovieLens
# 
### Исследование фильмов, оценок пользователей и тегов


In [ ]:
import re
from movielens_analysis import Movies, Links, Ratings, Tags

In [ ]:
movies = Movies("../datasets/movies.csv")
links = Links("../datasets/links.csv", cache_file="imdb_cache.json")
ratings = Ratings("../datasets/ratings.csv")
tags = Tags("../datasets/tags.csv")

print(f"Загружено фильмов: {len(movies.movies)}")
print(f"Загружено оценок: {len(ratings.ratings)}")
print(f"Загружено тегов: {len(tags.tags)}")

 ## 2. Анализ фильмов (Movies)

### 2.1 Распределение фильмов по годам выпуска

In [ ]:
years_distribution = movies.dist_by_release()
top_years = list(years_distribution.items())[:5]

print("Топ-5 лет по числу выпущенных фильмов:")
print("-" * 50)
for year, count in top_years:
    print(f"{year}: {count} фильмов")

 ### 2.2 Распределение фильмов по жанрам

In [ ]:
genres_distribution = movies.dist_by_genres()
total_movies = sum(genres_distribution.values())

print("Распределение фильмов по жанрам:")
print("-" * 60)
for genre, count in sorted(genres_distribution.items(), key=lambda x: x[1], reverse=True):
    percentage = (count / total_movies) * 100
    print(f"{genre:15} {count:6} ({percentage:.1f}%)")

### 2.3 Фильмы с наибольшим количеством жанров

In [ ]:
genre_masters = movies.most_genres(10)

print("Топ-10 фильмов с наибольшим числом жанров:")
print("-" * 50)
for i, (movie, count) in enumerate(list(genre_masters.items())[:5], 1):
    print(f"{i}. {movie[:60]}... ({count} жанров)")

## 3. Анализ IMDb-данных (Links)
### 3.1 Фильмы с наибольшим рейтингом

In [ ]:
rating_dist = links.find_all_highest_rated_movies()
movies_list, rating_value = rating_dist

print(f"Максимальный рейтинг IMDb: {rating_value}")
print("Фильмы с максимальным рейтингом:")
for movie in movies_list[:5]:
    title = movie.get("title", "Unknown")
    director = movie.get("director", "Unknown")
    print(f"  - {title} (реж. {director})")

### 3.2 Самые плодовитые режиссёры

In [ ]:
top_directors = links.top_directors(10, force_update=False)

print("Топ-10 режиссёров по числу фильмов:")
print("-" * 50)
for i, (director, count) in enumerate(top_directors.items(), 1):
    print(f"{i:2}. {director}: {count} фильмов")

### 3.3 Самые дорогие фильмы

In [ ]:
expensive_movies = links.most_expensive(10, force_update=False)

print("Топ-10 самых дорогих фильмов (бюджет):")
print("-" * 50)
for i, (movie, budget) in enumerate(list(expensive_movies.items())[:5], 1):
    print(f"{i}. {movie[:50]}... ${budget:,}")


### 3.4 Самые прибыльные фильмы

In [ ]:
profitable_movies = links.most_profitable(10, force_update=False)

print("Топ-10 самых прибыльных фильмов (сборы):")
print("-" * 50)
for i, (movie, profit) in enumerate(list(profitable_movies.items())[:5], 1):
    profit_millions = profit / 1_000_000
    print(f"{i}. {movie[:50]}... ${profit_millions:.1f} млн")


### 3.5 Самые длинные фильмы


In [ ]:
longest_movies = links.longest(10, force_update=False)
sorted_longest = sorted(longest_movies.items(), key=lambda x: x[1], reverse=True)

print("Топ-10 самых длинных фильмов (хронометраж):")
print("-" * 50)
for i, (movie, runtime) in enumerate(sorted_longest[:5], 1):
    hours = runtime // 60
    minutes = runtime % 60
    print(f"{i}. {movie[:50]}... ({hours}ч {minutes}м)")

runtimes = [runtime for _, runtime in sorted_longest[:10]]
if runtimes:
    avg_runtime = sum(runtimes) / len(runtimes)
    print(f"\nСредняя длительность топ-10: {avg_runtime:.1f} минут")

 ### 3.6 Стоимость минуты экранного времени

In [ ]:
cost_per_minute = links.top_cost_per_minute(10, force_update=False)

if cost_per_minute:
    sorted_cost = sorted(cost_per_minute.items(), key=lambda x: x[1], reverse=True)
    most_expensive_film, highest_cost = sorted_cost[0]
    print("Фильм с максимальной стоимостью минуты:")
    print(f"  '{most_expensive_film}'")
    print(f"  Стоимость: ${highest_cost/100:,.2f} за минуту")


## 4. Анализ оценок (Ratings)
### 4.1 Распределение оценок по годам


In [ ]:
years = ratings.movies.dist_by_year()
items = list(years.items())
peak_year, peak_count = max(items, key=lambda x: x[1])

print(f"Год с максимальным числом оценок: {peak_year} ({peak_count} оценок)")
print(f"Период данных: {items[0][0]} - {items[-1][0]}")

### 4.2 Распределение оценок по шкале


In [ ]:
dist = ratings.movies.dist_by_rating()

print("Распределение оценок:")
print("-" * 40)
for rating, cnt in sorted(dist.items()):
    print(f"{rating:.1f}: {cnt:6} оценок")

### 4.3 Средняя оценка

In [ ]:
bias = ratings.users.rating_bias()
print(f"Средняя оценка по всем фильмам: {bias:.2f}")

### 4.4 Популярные, любимые и спорные фильмы


In [ ]:
popular = ratings.movies.top_by_num_of_ratings(5)
loved = ratings.movies.top_by_ratings(5)
contro = ratings.movies.top_controversial(5)

print("Самые популярные (по числу оценок):")
for movie_id, count in popular.items():
    print(f"  Movie {movie_id}: {count} оценок")

print("\nСамые высоко оценённые:")
for movie_id, rating in loved.items():
    print(f"  Movie {movie_id}: {rating:.1f}")

print("\nСамые спорные (макс. дисперсия):")
for movie_id, var in contro.items():
    print(f"  Movie {movie_id}: дисперсия {var:.2f}")

### 4.5 Поляризующие фильмы (с учётом числа оценок)


In [ ]:
polarized = ratings.movies.most_polarized(5)

print("Топ-5 поляризующих фильмов (мин. 100 оценок):")
for movie_id, var in polarized.items():
    print(f"  Movie {movie_id}: дисперсия {var:.2f}")


 ## 5. Анализ пользователей (Users)
### 5.1 Распределение пользователей по активности

In [ ]:
activity = ratings.users.dist_by_num_of_ratings()

print("Распределение пользователей по числу оценок (первые 5):")
for num_ratings, count in list(activity.items())[:5]:
    print(f"  {num_ratings} оценок: {count} пользователей")


 ### 5.2 Распределение пользователей по среднему баллу


In [ ]:
styles = ratings.users.dist_by_ratings()

print("Распределение пользователей по среднему баллу (первые 5):")
for rating, count in list(styles.items())[:5]:
    print(f"  {rating:.2f}: {count} пользователей")


### 5.3 Самые непредсказуемые пользователи


In [ ]:
unstable = ratings.users.top_controversial(5)

print("Пользователи с максимальной дисперсией оценок:")
for user_id, var in unstable.items():
    print(f"  User {user_id}: дисперсия {var:.2f}")

## 6. Анализ тегов (Tags)
### 6.1 Самые многословные теги

In [ ]:
mw = tags.most_words(5)

print("Теги с максимальным числом слов:")
for tag, word_count in mw.items():
    print(f"  '{tag[:60]}' -> {word_count} слов")

### 6.2 Самые длинные теги

In [ ]:
lng = tags.longest(5)

print("Самые длинные теги (по символам):")
for tag, length in lng.items():
    print(f"  '{tag[:60]}' -> {length} символов")

### 6.3 Самые популярные теги

In [ ]:
pop = tags.most_popular(10)

print("Топ-10 самых частотных тегов:")
for tag, count in pop.items():
    print(f"  {tag}: {count} использований")

 ### 6.4 Частотность слов в тегах

In [ ]:
freq = tags.tag_word_frequency(10)

print("Топ-10 самых частотных слов в тегах:")
for word, count in freq.items():
    print(f"  {word}: {count}")

### 6.5 Поиск по ключевым словам

In [ ]:
good_tags = tags.tags_with("good")

print("Теги, содержащие слово 'good':")
for tag in good_tags[:10]:
    print(f"  {tag}")

## 7. Сводная статистика

In [ ]:
print("\nФильмы:")
print(f"  Всего фильмов в MovieLens: {len(movies.movies)}")
print(f"  Уникальных жанров: {len(genres_distribution)}")
print(f"  Самый популярный жанр: {max(genres_distribution.items(), key=lambda x: x[1])[0]}")

print("\nОценки:")
print(f"  Всего оценок: {len(ratings.ratings)}")
print(f"  Диапазон оценок: {min(dist.keys())} - {max(dist.keys())}")
print(f"  Средняя оценка: {bias:.2f}")

print("\nТеги:")
print(f"  Всего тегов: {len(tags.tags)}")
print(f"  Уникальных тегов: {len(tag.to_string() for tag in tags.tags)}")
print(f"  Самый частотный тег: {list(pop.items())[0][0]}")